# Zolai Loan Word LLM Filter
Filters English bleed-over from Zolai vocabulary while preserving legitimate loan words (like 'computer', 'motor'). Uses Google Gemini API.

In [ ]:
!pip install -q google-generativeai

import os
import json
import time
import urllib.request
import google.generativeai as genai

# Check internet connection
try:
    urllib.request.urlopen('http://google.com')
    print("Internet connection: OK")
except:
    print("Internet connection: FAILED. Please enable Internet in Kaggle Notebook settings.")


In [ ]:
# === SETUP ===
# Please add your Gemini API Key in Kaggle Secrets (Add-ons -> Secrets) as 'GEMINI_API_KEY'
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    genai.configure(api_key=api_key)
    print("Gemini API configured successfully.")
except Exception as e:
    print(f"Warning: Could not load GEMINI_API_KEY from secrets: {e}")
    print("You can manually set it below if not using Kaggle Secrets:")
    # genai.configure(api_key='YOUR_API_KEY')

import sys
sys.path.append('/kaggle/input/zolai-dataset-train/src')
# Assuming the dataset is mounted at /kaggle/input/zolai-dataset-train
DATA_DIR = "/kaggle/input/zolai-dataset-train/data"
FLAGGED_FILE = os.path.join(DATA_DIR, "flagged_foreign_words.json")
VOCAB_FILE = os.path.join(DATA_DIR, "zolai_unified_vocabulary.json")
OUTPUT_DIR = "/kaggle/working/"
LOAN_WORDS_FILE = os.path.join(OUTPUT_DIR, "legitimate_loan_words.json")
PURE_VOCAB_FILE = os.path.join(OUTPUT_DIR, "zolai_unified_vocabulary_pure.json")

In [ ]:
def get_system_prompt():
    return """You are a Zolai (Tedim Chin) language expert. 
Your task is to identify legitimate loan words from a list of flagged foreign/English words found in a Zolai dataset.
Legitimate loan words are words that Zolai speakers commonly use because there is no widely accepted native Zolai equivalent (e.g., 'computer', 'motor', 'plastic', 'phone', 'internet', 'radio', 'tv', 'video', 'bus', 'car', 'bank', 'office', 'school', 'hospital').
Do NOT include pure English bleed-over, OCR errors, HTML/CSS artifacts, or common English words that have a direct Zolai translation (e.g., 'max-width', 'px', 'the', 'and', 'x', 'javascript', 'is', 'book', 'house', 'water').
Return a JSON object with a single key "loan_words" containing a list of strings: ONLY the legitimate loan words from the input list."""

def process_batch_gemini(model_name, system_prompt, word_batch):
    model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=system_prompt,
        generation_config={"response_mime_type": "application/json", "temperature": 0.1}
    )
    prompt = f"Analyze these {len(word_batch)} words and return the legitimate Zolai loan words as a JSON array:\n{json.dumps(word_batch)}"
    try:
        response = model.generate_content(prompt)
        data = json.loads(response.text)
        return data.get("loan_words", [])
    except Exception as e:
        print(f"Error processing batch: {e}")
        return []

In [ ]:
# Load flagged words
if os.path.exists(FLAGGED_FILE):
    with open(FLAGGED_FILE, "r", encoding="utf-8") as f:
        flagged_data = json.load(f)
else:
    # Fallback to local test if not in Kaggle
    with open("../../data/flagged_foreign_words.json", "r", encoding="utf-8") as f:
        flagged_data = json.load(f)

sorted_words = sorted(flagged_data.keys(), key=lambda w: flagged_data[w].get("frequency", 0), reverse=True)
total_words = len(sorted_words)
print(f"Found {total_words} flagged words to process.")

In [ ]:
# Process using Gemini
BATCH_SIZE = 200
MODEL_NAME = "gemini-1.5-flash"
legitimate_loans = []
system_prompt = get_system_prompt()

for i in range(0, total_words, BATCH_SIZE):
    batch = sorted_words[i:i + BATCH_SIZE]
    print(f"Processing batch {i//BATCH_SIZE + 1}/{(total_words + BATCH_SIZE - 1)//BATCH_SIZE} ({len(batch)} words)...")
    
    loans = process_batch_gemini(MODEL_NAME, system_prompt, batch)
    print(f"  Found {len(loans)} loan words in this batch: {loans[:5]}{'...' if len(loans)>5 else ''}")
    legitimate_loans.extend(loans)
    
    # Save progress incrementally
    with open(LOAN_WORDS_FILE, "w", encoding="utf-8") as f:
        json.dump(legitimate_loans, f, ensure_ascii=False, indent=2)
        
    time.sleep(2)  # Rate limit pause

print(f"\nCompleted! Identified {len(legitimate_loans)} legitimate loan words.")

In [ ]:
# Rebuild Unified Vocabulary
if os.path.exists(VOCAB_FILE):
    with open(VOCAB_FILE, "r", encoding="utf-8") as f:
        vocab = json.load(f)
else:
    with open("../../data/zolai_unified_vocabulary.json", "r", encoding="utf-8") as f:
        vocab = json.load(f)

original_size = len(vocab)
# Keep words that are NOT flagged OR are in the legitimate loan words list
clean_vocab = {k: v for k, v in vocab.items() if k not in flagged_data or k in legitimate_loans}
new_size = len(clean_vocab)

with open(PURE_VOCAB_FILE, "w", encoding="utf-8") as f:
    json.dump(clean_vocab, f, ensure_ascii=False, indent=2)

print(f"Original vocabulary size: {original_size}")
print(f"Removed {original_size - new_size} flagged foreign/bleed words.")
print(f"Pure Zolai vocabulary size: {new_size}")
print(f"Saved pure vocabulary to {PURE_VOCAB_FILE}")